# House Price Prediction

In [73]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import OrderedDict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.impute  import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor , AdaBoostRegressor , RandomForestRegressor , StackingRegressor
from sklearn.linear_model import Ridge , RidgeCV , LassoCV , ElasticNetCV
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator, RegressorMixin
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from tensorflow.keras.layers import Dense , ELU , Dropout
from tensorflow.keras.regularizers import L1 , L2 , L1L2
from tensorflow.keras.losses import Huber
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from scikeras.wrappers import KerasRegressor

pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.2f}'.format

In [74]:
train_data = pd.read_csv(r"train.csv")
train_data.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.00,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.00,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.00,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.00,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.00,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [75]:
train_data.isna().sum()

Id                  0
MSSubClass          0
MSZoning            0
LotFrontage       259
LotArea             0
Street              0
Alley            1369
LotShape            0
LandContour         0
Utilities           0
LotConfig           0
LandSlope           0
Neighborhood        0
Condition1          0
Condition2          0
BldgType            0
HouseStyle          0
OverallQual         0
OverallCond         0
YearBuilt           0
YearRemodAdd        0
RoofStyle           0
RoofMatl            0
Exterior1st         0
Exterior2nd         0
MasVnrType        872
MasVnrArea          8
ExterQual           0
ExterCond           0
Foundation          0
BsmtQual           37
BsmtCond           37
BsmtExposure       38
BsmtFinType1       37
BsmtFinSF1          0
BsmtFinType2       38
BsmtFinSF2          0
BsmtUnfSF           0
TotalBsmtSF         0
Heating             0
HeatingQC           0
CentralAir          0
Electrical          1
1stFlrSF            0
2ndFlrSF            0
LowQualFin

In [76]:
test_data = pd.read_csv("test.csv")
test_data.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.00,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.00,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.00,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.00,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.00,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [77]:
test_data.shape

(1459, 80)

In [78]:
test_data.isna().sum()

Id                  0
MSSubClass          0
MSZoning            4
LotFrontage       227
LotArea             0
Street              0
Alley            1352
LotShape            0
LandContour         0
Utilities           2
LotConfig           0
LandSlope           0
Neighborhood        0
Condition1          0
Condition2          0
BldgType            0
HouseStyle          0
OverallQual         0
OverallCond         0
YearBuilt           0
YearRemodAdd        0
RoofStyle           0
RoofMatl            0
Exterior1st         1
Exterior2nd         1
MasVnrType        894
MasVnrArea         15
ExterQual           0
ExterCond           0
Foundation          0
BsmtQual           44
BsmtCond           45
BsmtExposure       44
BsmtFinType1       42
BsmtFinSF1          1
BsmtFinType2       42
BsmtFinSF2          1
BsmtUnfSF           1
TotalBsmtSF         1
Heating             0
HeatingQC           0
CentralAir          0
Electrical          0
1stFlrSF            0
2ndFlrSF            0
LowQualFin

### Remove Duplicate values(check Id)

In [79]:
duplicates = train_data['Id'].duplicated()
train_data[duplicates]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice


In [80]:
train_data.drop_duplicates(inplace=True)

### Remove values with no Id

In [81]:
no_id_data = train_data[train_data['Id'].isna()]
no_id_data

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice


In [82]:
train_data = train_data[train_data['Id'].notna()]
train_data.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.00,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.00,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.00,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.00,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.00,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


### Target Variables 

In [83]:
x = train_data.drop(columns = ['SalePrice' , 'Id'])
y = train_data['SalePrice']

#### Transform Y using log1p transformation (previously used yeo-jhonson transformation)

In [84]:
scaler = StandardScaler()

y_log1p = np.log1p(y)
y_log1p_scaled = scaler.fit_transform(np.array(y_log1p).reshape(-1,1))

### Important Variables

In [85]:
ordinal_cols = ['LotShape','Utilities','LandSlope','ExterQual','ExterCond',
                'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC','KitchenQual','Functional',
                'FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive','PoolQC','Fence']



nominal_cols = ['MSZoning','Street','Alley','Neighborhood','LotConfig','BldgType','Condition1','Condition2','HouseStyle','LandContour','RoofStyle','RoofMatl','Exterior1st' ,'Exterior2nd' ,
                'MasVnrType' , 'Foundation' , 'Heating' , 'GarageType' , 'MiscFeature' , 'SaleType' , 'SaleCondition', 'Electrical' , 'CentralAir']



impute_constant_cols = ['MSSubClass','MSZoning','Street','LotShape','LandContour','Utilities','LotConfig','LandSlope',
                        'Neighborhood','Condition1','Condition2','BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st',
                        'Exterior2nd','ExterQual','ExterCond','Foundation','Heating','HeatingQC','CentralAir','Electrical',
                        'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual','Fireplaces','GarageCars',
                        'PavedDrive','SaleType','SaleCondition', 'Functional']


# cols whose na values can be filled with median
numerical_cols = ['MSSubClass','LotArea','LotFrontage','OverallQual','OverallCond','YearBuilt','YearRemodAdd',
                  'MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','1stFlrSF','2ndFlrSF','LowQualFinSF',
                  'GrLivArea','BsmtFullBath','BsmtHalfBath','FullBath','HalfBath','BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd'
                  ,'Fireplaces','GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch','3SsnPorch'
                  ,'ScreenPorch','PoolArea','MiscVal','MoSold','YrSold']


# cols whose na values can be filled with default text values (like 'None' or 'No Garage')
# cols whose na values can be filled with 0
# (GarageYrBlt is kept here because if a house has no garage, a 0 is a safe placeholder before scaling/binarizing)
fill_default_cols = ['Alley','MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','FireplaceQu','GarageType',
                        'GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature' ,'GarageYrBlt']


default_map = {
    'Alley' : 'No alley',
    'MasVnrType' : 'None',
    'BsmtQual' : 'No Basement',
    'BsmtCond' : 'No Basement',
    'BsmtExposure' : 'No Basement',
    'BsmtFinType1' : 'No Basement',
    'BsmtFinType2' : 'No Basement',
    'FireplaceQu' : 'No Fireplace',
    'GarageType' : 'No Garage',
    'GarageFinish' : 'No Garage',
    'GarageQual' : 'No Garage',
    'GarageCond' : 'No Garage',
    'GarageYrBlt' : 0,
    'PoolQC' : 'No Pool',
    'Fence' : 'No Fence',
    'MiscFeature' : 'None'
}


ordinal_categories_map = {
'LotShape' : ['Missing' , 'IR3' , 'IR2' , 'IR1' , 'Reg'],
'Utilities' : ['Missing' , 'ELO' , 'NoSeWa' , 'NoSewr', 'AllPub'],
'LandSlope' : ['Missing' , 'Sev' , 'Mod' , 'Gtl'],
'ExterQual' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'ExterCond' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtQual' : ['No Basement' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtCond' : ['No Basement' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtExposure' : ['No Basement' , 'No' , 'Mn' , 'Av' , 'Gd'],
'BsmtFinType1' : ['No Basement' , 'Unf' , 'LwQ' , 'Rec' , 'BLQ' , 'ALQ' , 'GLQ'],
'BsmtFinType2' : ['No Basement' , 'Unf' , 'LwQ' , 'Rec' , 'BLQ' , 'ALQ' , 'GLQ'],
'HeatingQC' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'KitchenQual' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'Functional' :['Missing', 'Sal' , 'Sev' , 'Maj2' , 'Maj1' , 'Mod' , 'Min2' , 'Min1' , 'Typ'],
'FireplaceQu' : ['No Fireplace' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'GarageFinish' : ['No Garage' , 'Unf' , 'RFn' , 'Fin'],
'GarageQual' : ['No Garage' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'GarageCond' : ['No Garage' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'PavedDrive' : ['Missing' , 'N' , 'P' , 'Y'],
'PoolQC' : ['No Pool' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'Fence' : ['No Fence' , 'MnWw' , 'GdWo' , 'MnPrv' , 'GdPrv']
}


##different model Parameters
catboost_params = OrderedDict([('bagging_temperature',5.0), ('border_count',128), ('depth',6), ('iterations',5000), ('l2_leaf_reg',1), 
                               ('learning_rate',0.01), ('loss_function','RMSE'), ('random_state',42), ('task_type','GPU'), ('verbose',100)])
                              
trad_gbm_params = OrderedDict([('learning_rate', 0.1), ('loss', 'huber'), ('max_depth', 3), 
                               ('max_features', 'sqrt'), ('min_samples_split', 50), ('min_weight_fraction_leaf', 0.01),
                               ('n_estimators', 200), ('n_iter_no_change', 15), ('subsample', 1.0), 
                               ('validation_fraction', 0.15) , ('random_state',42)])

xgboost_params = OrderedDict([('colsample_bylevel', 0.6), ('colsample_bytree', 0.6), ('gamma', 0.1), 
                              ('grow_policy', 'depthwise'), ('learning_rate', 0.1), ('max_bin', 128), 
                              ('max_depth', 7), ('max_leaves', 127), ('min_child_weight', 7), 
                              ('reg_alpha', 0.5), ('reg_lambda', 5), ('subsample', 0.6) , ('random_state',42)])

lgbm_params = OrderedDict([('colsample_bynode', 0.8), ('colsample_bytree', 0.6), ('learning_rate', 0.05), 
                           ('max_bin', 512), ('max_depth', -1), ('min_child_samples', 20), 
                           ('min_child_weight', 10.0), ('min_split_gain', 0.1), ('n_estimators', 1000), 
                           ('num_leaves', 31), ('path_smooth', 5.0), ('reg_alpha', 0.1), ('reg_lambda', 1.0), 
                           ('subsample', 0.8), ('subsample_freq', 1) , ('random_state',42)])

adgbm_params = OrderedDict([('estimator' , DecisionTreeRegressor(max_depth=8 , min_samples_split=10)), 
                            ('learning_rate', 1.0), ('loss', 'square'), ('n_estimators', 500)])

rf_params = OrderedDict([('max_depth', 100), ('max_features', None), 
                         ('min_samples_split', 2), ('n_estimators', 300)])

### Fill Default value columns 

In [86]:
for col in fill_default_cols:
    default_val = default_map.get(col)
    train_data[col] = train_data[col].fillna(default_val)
    test_data[col] = test_data[col].fillna(default_val)
    print(f"Default Values of {col} filled with {default_val}")
    print(train_data[col].unique())
    print(train_data[col].unique())

Default Values of Alley filled with No alley
['No alley' 'Grvl' 'Pave']
['No alley' 'Grvl' 'Pave']
Default Values of MasVnrType filled with None
['BrkFace' 'None' 'Stone' 'BrkCmn']
['BrkFace' 'None' 'Stone' 'BrkCmn']
Default Values of BsmtQual filled with No Basement
['Gd' 'TA' 'Ex' 'No Basement' 'Fa']
['Gd' 'TA' 'Ex' 'No Basement' 'Fa']
Default Values of BsmtCond filled with No Basement
['TA' 'Gd' 'No Basement' 'Fa' 'Po']
['TA' 'Gd' 'No Basement' 'Fa' 'Po']
Default Values of BsmtExposure filled with No Basement
['No' 'Gd' 'Mn' 'Av' 'No Basement']
['No' 'Gd' 'Mn' 'Av' 'No Basement']
Default Values of BsmtFinType1 filled with No Basement
['GLQ' 'ALQ' 'Unf' 'Rec' 'BLQ' 'No Basement' 'LwQ']
['GLQ' 'ALQ' 'Unf' 'Rec' 'BLQ' 'No Basement' 'LwQ']
Default Values of BsmtFinType2 filled with No Basement
['Unf' 'BLQ' 'No Basement' 'ALQ' 'Rec' 'LwQ' 'GLQ']
['Unf' 'BLQ' 'No Basement' 'ALQ' 'Rec' 'LwQ' 'GLQ']
Default Values of FireplaceQu filled with No Fireplace
['No Fireplace' 'TA' 'Gd' 'Fa' 'Ex' '

### Numerical Pipeline

In [87]:
num_pipeline = Pipeline(
    steps=[
        ('imputer' , SimpleImputer(strategy='median')),
        ('scaler' , StandardScaler())
    ]
)

### Ordinal Pipeline

In [88]:
ord_categories = [ordinal_categories_map.get(col) for col in ordinal_categories_map.keys()]
ord_categories

[['Missing', 'IR3', 'IR2', 'IR1', 'Reg'],
 ['Missing', 'ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
 ['Missing', 'Sev', 'Mod', 'Gtl'],
 ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Basement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Basement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Basement', 'No', 'Mn', 'Av', 'Gd'],
 ['No Basement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
 ['No Basement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
 ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['Missing', 'Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
 ['No Fireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Garage', 'Unf', 'RFn', 'Fin'],
 ['No Garage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Garage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['Missing', 'N', 'P', 'Y'],
 ['No Pool', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Fence', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']]

In [89]:
ord_pipeline = Pipeline(
    steps=[
        ('imputer' , SimpleImputer(strategy='constant' , fill_value= 'Missing')),
        ('encoder' , OrdinalEncoder(categories= ord_categories , handle_unknown= 'use_encoded_value' , unknown_value=-1)),
        ('scaler' , StandardScaler())
    ]
)

### Nominal Pipeline

In [90]:
nom_pipeline = Pipeline(
    steps= [
        ('imputer' , SimpleImputer(strategy='constant' , fill_value='Missing')),
        ('encoder' , OneHotEncoder(handle_unknown='ignore' ,sparse_output = False)),
        ('scaler' , StandardScaler())
    ]
)

### Preprocessing Pipeline

In [91]:
preprocessing = ColumnTransformer(
    transformers= [
        ('num' , num_pipeline , numerical_cols),
        ('ord' , ord_pipeline , ordinal_cols),
        ('nom' , nom_pipeline , nominal_cols)
    ]
)

In [92]:
test_ids = test_data['Id']
test_data.drop(columns=['Id'] , inplace = True)

x = preprocessing.fit_transform(x)
test_data = preprocessing.transform(test_data)

##### Keras Model

In [93]:
def keras_model_fun():
    model = Sequential([
        Dense(64 , activation='relu' , kernel_initializer='he_normal' , bias_initializer= 'he_normal' ,kernel_regularizer = L1L2(0.01) , bias_regularizer= L1L2(0.01) ,input_shape = (226,)),
        ELU(alpha=0.01),
        Dropout(0.3),
        Dense(32 , activation='relu' , kernel_initializer='he_normal' , bias_regularizer= L1L2(0.01) ,kernel_regularizer = L1L2(0.01)),
        ELU(alpha = 0.01),
        Dense(1 , activation='linear')
    ])
    model.compile(
        optimizer = Adam(learning_rate=0.01),
        loss = Huber(),
        metrics = ['mean_squared_error']
    )
    
    return model



keras_scikit_model = KerasRegressor(
    model = keras_model_fun,
    epochs=1000,
    batch_size=32,
    verbose =  0
)

keras_model_wrapper = TransformedTargetRegressor(regressor=keras_scikit_model)

### Main Pipeline

In [94]:
catboost_model = CatBoostRegressor(**catboost_params)
trad_gbm_model = GradientBoostingRegressor(**trad_gbm_params)
xgbm_model = XGBRegressor(**xgboost_params)
lgbm_model = LGBMRegressor(**lgbm_params)
adgbm_model = AdaBoostRegressor(**adgbm_params)
rf_model = RandomForestRegressor(**rf_params)

alpha_range = np.logspace(-3,2,100)
ridge_model = RidgeCV(alphas = alpha_range)
lasso_model = LassoCV(alphas = alpha_range)
elastic_net_model = ElasticNetCV(alphas = alpha_range)

stack_model = StackingRegressor(
    estimators= [
        ('catboost', catboost_model),
        ('xg_boost' , xgbm_model),
        ('light_gbm' , lgbm_model),
        ('rf' , rf_model),
        ('trad_gbm' , trad_gbm_model)
        # ('keras_model' , keras_model_wrapper)
        # ('adaboost' , adgbm_model),
    ],
    final_estimator= elastic_net_model,
    cv = 5,
    n_jobs = 1,
    passthrough=False
)

stack_model.fit(x , y_log1p_scaled.ravel())

0:	learn: 0.9926592	total: 15.8ms	remaining: 1m 19s
100:	learn: 0.5658583	total: 1.27s	remaining: 1m 1s
200:	learn: 0.4010213	total: 2.66s	remaining: 1m 3s
300:	learn: 0.3330339	total: 4.04s	remaining: 1m 3s
400:	learn: 0.2995122	total: 5.67s	remaining: 1m 5s
500:	learn: 0.2800291	total: 7.03s	remaining: 1m 3s
600:	learn: 0.2673916	total: 8.3s	remaining: 1m
700:	learn: 0.2575019	total: 9.6s	remaining: 58.9s
800:	learn: 0.2500849	total: 10.9s	remaining: 57s
900:	learn: 0.2436013	total: 12.2s	remaining: 55.4s
1000:	learn: 0.2377960	total: 13.5s	remaining: 53.9s
1100:	learn: 0.2330649	total: 14.7s	remaining: 52.2s
1200:	learn: 0.2289161	total: 16.3s	remaining: 51.4s
1300:	learn: 0.2249357	total: 17.5s	remaining: 49.8s
1400:	learn: 0.2211434	total: 18.8s	remaining: 48.2s
1500:	learn: 0.2179208	total: 20s	remaining: 46.7s
1600:	learn: 0.2150698	total: 21.3s	remaining: 45.2s
1700:	learn: 0.2124701	total: 22.6s	remaining: 43.8s
1800:	learn: 0.2099124	total: 23.9s	remaining: 42.5s
1900:	learn:

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002441 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3730
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 141
[LightGBM] [Info] Start training from score 0.022285
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002658 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3695
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 143
[LightGBM] [Info] Start training from score -0.005409
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,estimators,"[('catboost', ...), ('xg_boost', ...), ...]"
,final_estimator,ElasticNetCV(...0000000e+02]))
,cv,5
,n_jobs,1
,passthrough,False
,verbose,0
,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,0.6


In [95]:
scaled_transformed_predictions = stack_model.predict(test_data)

inverse_scaled_transformed_predictions = scaler.inverse_transform(np.array(scaled_transformed_predictions).reshape(-1,1))
actual_predictions = np.expm1(inverse_scaled_transformed_predictions)
rounded_actual_predictions = [ round(x,4) for x in actual_predictions.flatten()]

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [96]:
results = pd.DataFrame({
    "Id" : test_ids,
    "SalePrice" : rounded_actual_predictions
})

results.to_csv("result.csv" , index=False)